# MIND Large Reranking - OPTIMIZED (GPU + Batching)

**Faster version with batch processing and vectorized operations**

Optimizations:
- Batch processing (1000 impressions at a time) instead of row-by-row
- Load behaviors upfront (more efficient than streaming)
- Vectorized numpy operations for scoring
- Efficient parquet batching (flush every 100K scores)
- GPU for embeddings (sentence-transformers)

Expected: ~2.5-3 hours (vs 4+ hours for row-by-row)

## Setup

In [1]:
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer

sys.path.insert(0, '/kaggle/input/datasets/kspsvlnsiddardha/mind-large-rerank-data/code')
from bm25_retrieval import InvertedIndex, tokenize
from fusion import weighted_fusion, scores_to_rank_permutation

print("✓ Imports successful")

✓ Imports successful


## Parameters

In [2]:
ALPHA = 0.5
BATCH_SIZE = 1000

DATA_DIR = Path('/kaggle/input/datasets/kspsvlnsiddardha/mind-large-rerank-data')
SCORES_PATH = DATA_DIR / 'scores' / 'scores.parquet'
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

HAVE_CACHED_SCORES = SCORES_PATH.exists()

print(f"ALPHA: {ALPHA}, BATCH_SIZE: {BATCH_SIZE}")
print(f"Cached scores: {HAVE_CACHED_SCORES}")

ALPHA: 0.5, BATCH_SIZE: 1000
Cached scores: False


## Full Pipeline - OPTIMIZED

In [3]:
if not HAVE_CACHED_SCORES:
    print("[1/4] Loading articles and building BM25 index...")
    articles_path = DATA_DIR / 'data' / 'news.tsv'
    articles_df = pd.read_csv(articles_path, sep='\t', header=None,
                               names=['article_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'entities_title', 'entities_body'])
    
    article_text = {}
    for _, row in articles_df.iterrows():
        aid = row['article_id']
        text = (str(row['title']) or "") + " " + (str(row['abstract']) or "")
        article_text[aid] = text.strip()
    
    index = InvertedIndex.build(articles_df[['article_id', 'title', 'abstract']].copy())
    print(f"  ✓ BM25 index: {len(index.postings):,} terms, {index.N:,} documents")

[1/4] Loading articles and building BM25 index...
  ✓ BM25 index: 80,102 terms, 120,959 documents


In [4]:
if not HAVE_CACHED_SCORES:
    print("\n[2/4] Building embeddings (GPU)...")
    model = SentenceTransformer('BAAI/bge-base-en-v1.5')
    
    texts = [(str(row['title']) or "") + " " + (str(row['abstract']) or "") for _, row in articles_df.iterrows()]
    texts = [t.strip() or "[empty]" for t in texts]
    
    embeddings = model.encode(texts, batch_size=128, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)
    embeddings = embeddings.astype(np.float32)
    print(f"  ✓ Embeddings: {embeddings.shape}")
    
    aid_to_row = {aid: i for i, aid in enumerate(articles_df['article_id'])}
    del model


[2/4] Building embeddings (GPU)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/945 [00:00<?, ?it/s]

  ✓ Embeddings: (120959, 768)


In [5]:
if not HAVE_CACHED_SCORES:
    print("\n[3/4] Loading behaviors (2.37M impressions)...")
    behaviors_path = DATA_DIR / 'data' / 'behaviors.tsv'
    
    behaviors_list = []
    with open(behaviors_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 5:
                impression_id = parts[0]
                history_ids = parts[3].split() if parts[3] else []
                candidate_ids = parts[4].split() if parts[4] else []
                if candidate_ids:
                    behaviors_list.append((impression_id, history_ids, candidate_ids))
    
    print(f"  ✓ Loaded {len(behaviors_list):,} impressions")


[3/4] Loading behaviors (2.37M impressions)...
  ✓ Loaded 2,370,727 impressions


In [6]:
if not HAVE_CACHED_SCORES:
    print("\n[4/4] Batch processing and scoring...")
    
    scores_out_path = OUT_DIR / 'scores.parquet'
    schema = pa.schema([
        ('impression_id', pa.string()),
        ('article_id', pa.string()),
        ('bm25_score', pa.float32()),
        ('semantic_score', pa.float32())
    ])
    writer = pq.ParquetWriter(str(scores_out_path), schema)
    pred_file = open(OUT_DIR / 'predictions.txt', 'w')
    
    score_rows = []
    pred_lines = 0
    
    for batch_start in range(0, len(behaviors_list), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(behaviors_list))
        
        for impression_id, history_ids, candidate_ids in behaviors_list[batch_start:batch_end]:
            # BM25 scores
            bm25_scores = {}
            if history_ids:
                query_text = " ".join(article_text.get(aid, "") for aid in history_ids)
                qtf = Counter(tokenize(query_text))
                if qtf:
                    bm25_scores = index.score_documents_batch(set(candidate_ids), qtf, k1=1.5, b=0.75)
            for aid in candidate_ids:
                if aid not in bm25_scores:
                    bm25_scores[aid] = 0.0
            
            # Semantic scores
            semantic_scores = {}
            if history_ids:
                history_embs = [embeddings[aid_to_row[aid]] for aid in history_ids if aid in aid_to_row]
                query_emb = np.mean(history_embs, axis=0) if history_embs else np.zeros(embeddings.shape[1], dtype=np.float32)
            else:
                query_emb = np.zeros(embeddings.shape[1], dtype=np.float32)
            
            query_norm = np.linalg.norm(query_emb)
            if query_norm > 1e-8:
                query_normalized = query_emb / query_norm
                for aid in candidate_ids:
                    if aid in aid_to_row:
                        semantic_scores[aid] = float(np.dot(query_normalized, embeddings[aid_to_row[aid]]))
                    else:
                        semantic_scores[aid] = 0.0
            else:
                semantic_scores = {aid: 0.0 for aid in candidate_ids}
            
            # Save scores and predictions
            for aid in candidate_ids:
                score_rows.append({
                    'impression_id': impression_id,
                    'article_id': aid,
                    'bm25_score': bm25_scores.get(aid, 0.0),
                    'semantic_score': semantic_scores.get(aid, 0.0)
                })
            
            retriever_scores = {"bm25": bm25_scores, "semantic": semantic_scores}
            fused = weighted_fusion(candidate_ids, retriever_scores, {"bm25": 1 - ALPHA, "semantic": ALPHA})
            ranks = scores_to_rank_permutation(candidate_ids, fused)
            pred_file.write(f"{impression_id} [{','.join(str(r) for r in ranks)}]\n")
            pred_lines += 1
        
        if len(score_rows) >= 100000:
            writer.write_table(pa.Table.from_pylist(score_rows, schema=schema))
            score_rows = []
        
        if batch_end % 100000 <= BATCH_SIZE:
            print(f"  Processed {batch_end:,} impressions, {pred_lines:,} predictions")
    
    if score_rows:
        writer.write_table(pa.Table.from_pylist(score_rows, schema=schema))
    
    writer.close()
    pred_file.close()
    print(f"\n✓ Wrote {pred_lines:,} predictions")


[4/4] Batch processing and scoring...
  Processed 1,000 impressions, 1,000 predictions
  Processed 100,000 impressions, 100,000 predictions
  Processed 101,000 impressions, 101,000 predictions
  Processed 200,000 impressions, 200,000 predictions
  Processed 201,000 impressions, 201,000 predictions
  Processed 300,000 impressions, 300,000 predictions
  Processed 301,000 impressions, 301,000 predictions
  Processed 400,000 impressions, 400,000 predictions
  Processed 401,000 impressions, 401,000 predictions
  Processed 500,000 impressions, 500,000 predictions
  Processed 501,000 impressions, 501,000 predictions
  Processed 600,000 impressions, 600,000 predictions
  Processed 601,000 impressions, 601,000 predictions
  Processed 700,000 impressions, 700,000 predictions
  Processed 701,000 impressions, 701,000 predictions
  Processed 800,000 impressions, 800,000 predictions
  Processed 801,000 impressions, 801,000 predictions
  Processed 900,000 impressions, 900,000 predictions
  Processed

## Verify

In [7]:
pred_out = OUT_DIR / 'predictions.txt'
with open(pred_out) as f:
    n_lines = sum(1 for _ in f)
print(f"✓ predictions.txt: {n_lines:,} lines (expect 2,370,727)")
if n_lines == 2370727:
    print("  ✓✓✓ COMPLETE!")

import zipfile
with zipfile.ZipFile(OUT_DIR / 'predictions.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_out, arcname='predictions.txt')
print(f"✓ Created predictions.zip")

✓ predictions.txt: 2,370,727 lines (expect 2,370,727)
  ✓✓✓ COMPLETE!
✓ Created predictions.zip
